# 🌐 SAST/SCA Prototype — Stage: NVD Data Collection
**MTech Project | SCA — Dependency Risk Scoring**

```
NVD REST API  →  CVE-level features
PyPI / Maven  →  ecosystem-level metadata
                        ↓
              sca_dataset.csv
                        ↓
              06_xgboost_sca_model.ipynb
```

This notebook builds the SCA training dataset described in **Chapter 4.2 / 4.4** of the
dissertation: it pulls CVE records for the Python (PyPI) and Java (Maven) ecosystems from
the **NIST NVD REST API**, enriches them with package-registry metadata, and writes out
`sca_dataset.csv` in the schema from **Table 4.2**.

> ⚠️ Requires outbound internet access to `services.nvd.nist.gov`, `pypi.org`, and
> `search.maven.org`. Get a free NVD API key at
> https://nvd.nist.gov/developers/request-an-api-key — without one you're rate-limited to
> 5 requests / 30s (with a key: 50 requests / 30s), which matters a lot when pulling
> ~45,000 records.

## 📦 1. Imports & Config

In [ ]:
import os
import time
import math
import json
import requests
import numpy as np
import pandas as pd
from datetime import datetime, timezone
from dateutil import parser as dateparser

NVD_API_KEY = os.environ.get("", "")   # set this env var if you have a key
NVD_BASE_URL = "https://services.nvd.nist.gov/rest/json/cves/2.0"

# Keywords used to filter CVEs down to the Python/Java package ecosystems we care about
ECOSYSTEM_KEYWORDS = ["pypi", "python", "maven", "java", "npm", "log4j", "spring"]

RESULTS_PER_PAGE = 200          # NVD max page size
REQUEST_DELAY_SEC = 6 if not NVD_API_KEY else 0.6   # stay under the rate limit
MAX_RECORDS = 45_000             # matches Table 3.1 / Chapter 5.2 dataset size

print(f"✅ Config ready. API key set: {bool(NVD_API_KEY)} | Target records: {MAX_RECORDS:,}")

✅ Config ready. API key set: True | Target records: 45,000


In [7]:
import os

os.environ["REQUESTS_CA_BUNDLE"] = r"C:\Users\Tejal\miniconda3\Library\ssl\cacert.pem"

In [8]:
import os
print(os.environ.get("REQUESTS_CA_BUNDLE"))

C:\Users\Tejal\miniconda3\Library\ssl\cacert.pem


## 📥 2. Pull CVE Records from NVD

Pages through `/cves/2.0`, filtering to CVEs published in the last N months whose
description mentions one of the target ecosystems. NVD's `keywordSearch` parameter does
a substring match on the CVE description, which is a coarse filter — the enrichment step
(§4) does the real ecosystem confirmation by hitting PyPI/Maven for each candidate
package name.

In [9]:
def fetch_nvd_page(start_index: int, keyword: str, results_per_page: int = RESULTS_PER_PAGE) -> dict:
    headers = {"apiKey": NVD_API_KEY} if NVD_API_KEY else {}
    params = {
        "startIndex": start_index,
        "resultsPerPage": results_per_page,
        "keywordSearch": keyword,
    }
    resp = requests.get(NVD_BASE_URL, headers=headers, params=params, timeout=30)
    resp.raise_for_status()
    return resp.json()


def collect_cves(keywords, max_records: int = MAX_RECORDS) -> list:
    """Pages through NVD for each keyword until max_records is reached or exhausted."""
    all_records, seen_ids = [], set()

    for kw in keywords:
        start_index = 0
        while len(all_records) < max_records:
            try:
                page = fetch_nvd_page(start_index, kw)
            except requests.exceptions.RequestException as e:
                print(f"  ⚠️  Request failed for '{kw}' @ {start_index}: {e}. Backing off...")
                time.sleep(10)
                continue

            vulns = page.get("vulnerabilities", [])
            if not vulns:
                break

            for v in vulns:
                cve_id = v["cve"]["id"]
                if cve_id not in seen_ids:
                    seen_ids.add(cve_id)
                    all_records.append(v["cve"])

            total_results = page.get("totalResults", 0)
            start_index += RESULTS_PER_PAGE
            print(f"  [{kw}] {len(all_records):,}/{max_records:,} collected "
                  f"(NVD total for this keyword: {total_results:,})", end="\r")

            time.sleep(REQUEST_DELAY_SEC)
            if start_index >= total_results:
                break

        if len(all_records) >= max_records:
            break

    print()
    return all_records[:max_records]


# NOTE: a full 45k pull takes hours even with an API key. For a smoke-test run,
# lower MAX_RECORDS above (e.g. 500) before executing this cell.
raw_cves = collect_cves(ECOSYSTEM_KEYWORDS, max_records=MAX_RECORDS)
print(f"\n✅ Collected {len(raw_cves):,} raw CVE records")

  [spring] 14,683/45,000 collected (NVD total for this keyword: 453))

✅ Collected 14,683 raw CVE records


## 🧮 3. Extract CVE-Level Features (Table 4.2 — cvss_*, has_exploit)

In [10]:
def extract_cve_features(cve: dict) -> dict:
    cve_id = cve["id"]
    published = cve.get("published")
    descriptions = cve.get("descriptions", [])
    desc_text = next((d["value"] for d in descriptions if d.get("lang") == "en"), "")

    # CVSS v3.1 (fallback to v3.0, then v2 if absent — NVD coverage varies by CVE age)
    metrics = cve.get("metrics", {})
    cvss_block = (metrics.get("cvssMetricV31") or metrics.get("cvssMetricV30")
                  or metrics.get("cvssMetricV2") or [])
    cvss_data = cvss_block[0]["cvssData"] if cvss_block else {}

    base_score = cvss_data.get("baseScore", np.nan)
    exploitability = (cvss_block[0].get("exploitabilityScore", np.nan) if cvss_block else np.nan)
    severity = cvss_data.get("baseSeverity", cvss_block[0].get("baseSeverity") if cvss_block else None)

    # Known-exploited: NVD flags references tagged "Exploit"
    references = cve.get("references", [])
    has_exploit = int(any("Exploit" in ref.get("tags", []) for ref in references))

    # CWE classification (first listed weakness)
    weaknesses = cve.get("weaknesses", [])
    cwe_id = None
    if weaknesses:
        cwe_desc = weaknesses[0].get("description", [])
        cwe_id = next((d["value"] for d in cwe_desc if d.get("lang") == "en"), None)

    # Extract a probable package name from the description (crude heuristic —
    # refined later in the enrichment step against real PyPI/Maven records)
    package_guess = None
    for token in desc_text.replace(",", " ").split():
        if len(token) > 2 and token.islower() and token.isascii() and token.isalpha():
            package_guess = token
            break

    return {
        "cve_id": cve_id,
        "published": published,
        "description": desc_text[:300],
        "cvss_base_score": base_score,
        "cvss_exploitability": exploitability,
        "severity_label": severity,
        "has_exploit": has_exploit,
        "cwe_id": cwe_id,
        "package_guess": package_guess,
    }


cve_df = pd.DataFrame([extract_cve_features(c) for c in raw_cves])
cve_df = cve_df.dropna(subset=["severity_label"]).reset_index(drop=True)
print(f"✅ Extracted features for {len(cve_df):,} CVEs with a usable severity label")
cve_df.head()

✅ Extracted features for 14,279 CVEs with a usable severity label


,cve_id,published,description,cvss_base_score,cvss_exploitability,severity_label,has_exploit,cwe_id,package_guess
0,CVE-2013-1629,2013-08-06T02:52:10.177,pip before 1.3 uses HTTP to retrieve packages ...,6.8,8.6,MEDIUM,1,CWE-20,pip
1,CVE-2013-1630,2013-08-06T02:52:10.223,pyshop before 0.7.1 uses HTTP to retrieve pack...,6.8,8.6,MEDIUM,0,CWE-20,pyshop
2,CVE-2013-1633,2013-08-06T02:52:10.333,easy_install in setuptools before 0.7 uses HTT...,6.8,8.6,MEDIUM,0,CWE-20,setuptools
3,CVE-2019-6802,2019-01-25T04:29:00.240,CRLF Injection in pypiserver 1.2.5 and below a...,6.1,2.8,MEDIUM,1,CWE-74,pypiserver
4,CVE-2020-13328,2020-09-30T18:15:19.833,An issue has been discovered in GitLab affecti...,4.8,1.7,MEDIUM,1,CWE-79,issue


## 📦 4. Enrich with PyPI / Maven Registry Metadata (Chapter 4.4)

For each candidate package, pull ecosystem metadata to compute `days_since_patch`,
`patch_lag_days`, `dependency_depth`, and `maintainer_activity`. This calls the public
PyPI JSON API and the Maven Central search API — no auth required, but be polite with
rate limiting (both throttle aggressive scraping).

In [12]:
def _to_utc(dt: datetime) -> datetime:
    """Normalise a datetime to timezone-aware UTC (NVD and PyPI don't always agree
    on whether their timestamps carry an explicit offset)."""
    if dt.tzinfo is None:
        return dt.replace(tzinfo=timezone.utc)
    return dt.astimezone(timezone.utc)


def fetch_pypi_metadata(package: str) -> dict | None:
    try:
        resp = requests.get(f"https://pypi.org/pypi/{package}/json", timeout=10)
        if resp.status_code != 200:
            return None
        data = resp.json()
        releases = data.get("releases", {})
        upload_times = [
            _to_utc(dateparser.isoparse(r["upload_time_iso_8601"]))
            for rel in releases.values() for r in rel if r.get("upload_time_iso_8601")
        ]
        if not upload_times:
            return None
        upload_times.sort()
        return {
            "latest_release": upload_times[-1],
            "release_count_12m": sum(
                1 for t in upload_times
                if (datetime.now(timezone.utc) - t).days <= 365
            ),
            "total_releases": len(upload_times),
        }
    except requests.exceptions.RequestException:
        return None


def enrich_row(row: pd.Series) -> pd.Series:
    pkg = row["package_guess"]
    meta = fetch_pypi_metadata(pkg) if pkg else None

    if meta and row["published"]:
        published_dt = _to_utc(dateparser.isoparse(row["published"]))
        days_since_patch = max((meta["latest_release"] - published_dt).days, 0)
        maintainer_activity = round(meta["release_count_12m"] / 12, 2)   # releases/month, normalised later
        dependency_depth = min(meta["total_releases"] // 20, 10)          # crude proxy, capped
    else:
        # Package not resolvable on PyPI (likely a Maven/Java package, or guess failed) —
        # fall back to NaN; imputed in the next notebook per Chapter 4.4's stated approach.
        days_since_patch = np.nan
        maintainer_activity = np.nan
        dependency_depth = np.nan

    row["days_since_patch"] = days_since_patch
    row["maintainer_activity"] = maintainer_activity
    row["dependency_depth"] = dependency_depth
    return row


# Enrichment is one HTTP call per row — throttle to stay well within PyPI's fair-use limits.
enriched_rows = []
for i, row in cve_df.iterrows():
    enriched_rows.append(enrich_row(row.copy()))
    if i % 50 == 0:
        print(f"  enriched {i:,}/{len(cve_df):,}", end="\r")
    time.sleep(0.2)

cve_df = pd.DataFrame(enriched_rows)
print(f"\n✅ Enrichment complete. PyPI-resolved: {cve_df['days_since_patch'].notna().sum():,}/{len(cve_df):,}")

  enriched 14,250/14,279
✅ Enrichment complete. PyPI-resolved: 6,570/14,279


## 🧾 5. Derive Remaining Table 4.2 Features

In [15]:
# vuln_count_12m — how many other CVEs share this package_guess and were published
# in the trailing 12 months (a simple co-occurrence count within the collected sample).
cve_df["published_dt"] = pd.to_datetime(cve_df["published"], errors="coerce", utc=True)

def count_recent_vulns(row, window_days=365):
    if pd.isna(row["published_dt"]) or not row["package_guess"]:
        return np.nan
    mask = (
        (cve_df["package_guess"] == row["package_guess"]) &
        (cve_df["published_dt"] <= row["published_dt"]) &
        (cve_df["published_dt"] >= row["published_dt"] - pd.Timedelta(days=window_days))
    )
    return int(mask.sum())

cve_df["vuln_count_12m"] = cve_df.apply(count_recent_vulns, axis=1)

# patch_lag_days — days between CVE publish date and (proxy) patch release date.
# NVD does not expose a clean "patched version date" field directly, so this uses the
# same PyPI release lookup as days_since_patch as the closest available proxy.
cve_df["patch_lag_days"] = cve_df["days_since_patch"]

print("✅ Derived vuln_count_12m and patch_lag_days")
cve_df[["cve_id", "cvss_base_score", "cvss_exploitability", "has_exploit",
        "vuln_count_12m", "days_since_patch", "patch_lag_days",
        "dependency_depth", "maintainer_activity", "severity_label"]].head()

✅ Derived vuln_count_12m and patch_lag_days


,cve_id,cvss_base_score,cvss_exploitability,has_exploit,vuln_count_12m,days_since_patch,patch_lag_days,dependency_depth,maintainer_activity,severity_label
0,CVE-2013-1629,6.8,8.6,1,1.0,4740.0,4740.0,10.0,1.17,MEDIUM
1,CVE-2013-1630,6.8,8.6,0,1.0,1423.0,1423.0,1.0,0.00,MEDIUM
2,CVE-2013-1633,6.8,8.6,0,1.0,4715.0,4715.0,10.0,1.33,MEDIUM
3,CVE-2019-6802,6.1,2.8,1,1.0,2573.0,2573.0,5.0,0.33,MEDIUM
4,CVE-2020-13328,4.8,1.7,1,65.0,504.0,504.0,0.0,0.00,MEDIUM


## 💾 6. Save `sca_dataset.csv`

In [16]:
SCA_COLUMNS = [
    "cve_id", "package_guess", "cwe_id",
    "cvss_base_score", "cvss_exploitability",
    "vuln_count_12m", "days_since_patch", "patch_lag_days",
    "has_exploit", "dependency_depth", "maintainer_activity",
    "severity_label",
]

sca_dataset = cve_df[SCA_COLUMNS].copy()
sca_dataset.to_csv("sca_dataset.csv", index=False)

severity_counts = sca_dataset["severity_label"].value_counts().to_dict()
missing_exploit = sca_dataset["has_exploit"].isna().sum()
missing_patch_lag = sca_dataset["patch_lag_days"].isna().sum()

print("SCA Dataset Summary")
print("-" * 40)
print(f"Records            : {len(sca_dataset):,}")
print(f"Severity classes   : {severity_counts}")
print(f"Missing has_exploit: {missing_exploit}")
print(f"Missing patch_lag  : {missing_patch_lag} "
      "(will be imputed with the 95th percentile in 06_xgboost_sca_model.ipynb, per Chapter 4.4)")
print()
print("✅ Saved → sca_dataset.csv")
print("Next: run 06_xgboost_sca_model.ipynb")

SCA Dataset Summary
----------------------------------------
Records            : 14,279
Severity classes   : {'MEDIUM': 8559, 'HIGH': 3927, 'CRITICAL': 1283, 'LOW': 504, 'NONE': 6}
Missing has_exploit: 0
Missing patch_lag  : 7709 (will be imputed with the 95th percentile in 06_xgboost_sca_model.ipynb, per Chapter 4.4)

✅ Saved → sca_dataset.csv
Next: run 06_xgboost_sca_model.ipynb
